# Data Preparation: College Scorecard, ACS, and FSA (2011–2020)

This notebook documents the full data-preparation pipeline for the student loan project. It takes three raw public data sources — the U.S. Department of Education College Scorecard data, IPUMS ACS microdata, and FSA FAFSA application volume by state — and converts them into consistent, analysis-ready state–year datasets.

Specifically, the notebook:
- Describes how to download each raw source file (College Scorecard, ACS extract from IPUMS, and FAFSA-by-state Excel files).
- Cleans and filters each dataset, selecting the variables needed for the project and harmonizing state and year identifiers.
- Aggregates ACS household/person-level data to state–year controls (income, poverty, enrollment, unemployment, etc.).
- Reshapes and annualizes FSA application counts by state and cycle.
- Produces cleaned state–year CSVs for each source and merges them into a single panel covering 2011–2020.

The resulting combined dataset serves as the starting point for the project’s exploratory analysis and regression modeling.


# 1. College Scorecard Cleanup

## Source for college score card:  https://collegescorecard.ed.gov/data
1. click the darkgreen button with "download(.zip,450MB)" next to "All Data Files", zip file name is College_Scorecard_Raw_Data_10032025.zip.
2. merge all the MERGEDYYYY_YY_PP.csv files as our combined collge scorecard data.

**See pdf version of the steps in  SourceFiles_download_instruction.pdf**


In [2]:
library(readr)
library(dplyr)
library(stringr)
library(purrr)

In [19]:
scorecard_dir <- "College_Scorecard_Raw_Data_05192025/"

# 1) list MERGED files only (panel)
# match the pattern MERGEDYYYY_YY_PP.csv, an
files <- list.files(scorecard_dir, pattern = "^MERGED\\d{4}_\\d{2}_PP\\.csv$", full.names = TRUE)

# helper: robust numeric conversion for Scorecard sentinels
# to_num <- function(x) suppressWarnings(as.numeric(na_if(na_if(x, "NULL"), "PrivacySuppressed")))
# Robust converter: works whether values are "NULL", "PrivacySuppressed", numbers, or blanks
to_num <- function(x) {
  x <- as.character(x)
  x <- na_if(x, "NULL")
  x <- na_if(x, "PrivacySuppressed")
  x <- na_if(x, "")
  suppressWarnings(as.numeric(x))
}

# columns to keep (add if  want more controls like ADM_RATE, SAT_AVG, C150_4)
# keep <- c("UNITID","INSTNM","STABBR","UGDS","CONTROL",
#           "DEBT_MDN","PCTPELL","NPT4_PUB","NPT4_PRIV",
#           # STEM program shares:
#           "PCIP11","PCIP14","PCIP15","PCIP26","PCIP27","PCIP40")
keep <- c(
  "UNITID","INSTNM","STABBR","UGDS","CONTROL",
  "DEBT_MDN","PCTPELL","NPT4_PUB","NPT4_PRIV",
  "PCIP11","PCIP14","PCIP15","PCIP26","PCIP27","PCIP40"
)

In [10]:
# 2) read and stack, attaching YEAR from filename
# sc_all <- map_dfr(files, read_one)
read_one <- function(path){
  yr <- stringr::str_match(basename(path), "^MERGED(\\d{4})_\\d{2}_PP\\.csv$")[,2] |> as.integer()
  readr::read_csv(
    path,
    col_select = any_of(keep),
    col_types  = cols(.default = col_character()),  # <<< force all as character
    show_col_types = FALSE
  ) |>
    mutate(
      YEAR      = yr,
      UGDS      = to_num(UGDS),
      CONTROL   = as.integer(CONTROL),
      DEBT_MDN  = to_num(DEBT_MDN),
      PCTPELL   = to_num(PCTPELL),
      NPT4_PUB  = to_num(NPT4_PUB),
      NPT4_PRIV = to_num(NPT4_PRIV),
      across(starts_with("PCIP"), to_num)
    )
}

sc_all <- purrr::map_dfr(files, read_one)

In [11]:
# 3) institution STEM share (sum PCIP’s; they are proportions 0–1)
sc_all <- sc_all |>
  rowwise() |>
  mutate(stem_share_inst = sum(c_across(c(PCIP11,PCIP14,PCIP15,PCIP26,PCIP27,PCIP40)), na.rm = TRUE)) |>
  ungroup()

# 4) weighted state–year aggregates
wavg <- function(x, w){
  ok <- is.finite(x) & is.finite(w) & w > 0
  if (!any(ok)) return(NA_real_)
  sum(x[ok] * w[ok]) / sum(w[ok])
}

scorecard_state_year <- sc_all |>
  filter(!is.na(STABBR), !is.na(YEAR), UGDS > 0) |>
  group_by(STABBR, YEAR) |>
  summarise(
    ugds_total      = sum(UGDS, na.rm = TRUE),
    ugds_public     = sum(UGDS[CONTROL == 1], na.rm = TRUE),
    ugds_private_np = sum(UGDS[CONTROL == 2], na.rm = TRUE),
    ugds_forprofit  = sum(UGDS[CONTROL == 3], na.rm = TRUE),

    grad_debt_mdn_w  = wavg(DEBT_MDN, UGDS),     # <-- outcome
    state_pell_share = wavg(PCTPELL, UGDS),

    npt_pub_w  = wavg(NPT4_PUB,  UGDS),
    npt_priv_w = wavg(NPT4_PRIV, UGDS),

    # overall net price (weight sector means by sector enrollments)
    state_net_price = {
      num <- npt_pub_w * ugds_public + npt_priv_w * ugds_private_np + npt_priv_w * ugds_forprofit
      den <- ugds_public + ugds_private_np + ugds_forprofit
      ifelse(den > 0, num / den, NA_real_)
    },

    sector_public_share     = ifelse(ugds_total > 0, ugds_public     / ugds_total, NA_real_),
    sector_private_np_share = ifelse(ugds_total > 0, ugds_private_np / ugds_total, NA_real_),
    sector_forprofit_share  = ifelse(ugds_total > 0, ugds_forprofit  / ugds_total, NA_real_),

    state_stem_share = wavg(stem_share_inst, UGDS),

    .groups = "drop"
  ) |>
  mutate(
    state = STABBR,
    log_grad_debt_state = log(grad_debt_mdn_w)
  ) |>
  select(state, YEAR, log_grad_debt_state, grad_debt_mdn_w, state_net_price,
         state_pell_share, starts_with("sector_"), state_stem_share, ugds_total)

# 5) sanity check & save
print(head(scorecard_state_year, 10), width = Inf)
readr::write_csv(scorecard_state_year, "scorecard_state_year.csv")


# A tibble: 10 x 11
   state  YEAR log_grad_debt_state grad_debt_mdn_w state_net_price
   <chr> <int>               <dbl>           <dbl>           <dbl>
 1 AK     1996               NA                NA               NA
 2 AK     1997                8.46           4726.              NA
 3 AK     1998                8.56           5196.              NA
 4 AK     1999                8.51           4964.              NA
 5 AK     2001                8.43           4560.              NA
 6 AK     2002                8.36           4271.              NA
 7 AK     2003                8.40           4429.              NA
 8 AK     2004                8.46           4729.              NA
 9 AK     2005                8.54           5119.              NA
10 AK     2006                8.50           4919.              NA
   state_pell_share sector_public_share sector_private_np_share
              <dbl>               <dbl>                   <dbl>
 1               NA               0.938         

In [12]:
dim(scorecard_state_year)

[1] 1593   11

In [13]:
summary(sc_all$PCTPELL)
if (max(sc_all$PCTPELL, na.rm = TRUE) > 1) {
  sc_all$PCTPELL <- sc_all$PCTPELL / 100
}


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
   0.00    0.30    0.46    0.48    0.65    1.00   93622 

In [14]:
# Sanity check
library(dplyr)

# Coverage by year
scorecard_state_year %>%
  count(YEAR) %>%
  arrange(YEAR)

# Fraction non-missing by year for key variables
scorecard_state_year %>%
  group_by(YEAR) %>%
  summarise(
    n = n(),
    avail_debt   = mean(!is.na(grad_debt_mdn_w)),
    avail_price  = mean(!is.na(state_net_price)),
    avail_pell   = mean(!is.na(state_pell_share))
  ) %>%
  arrange(YEAR)

# Keep modern years with decent coverage
scorecard_state_year_clean <- scorecard_state_year %>%
  filter(YEAR >= 2011)

# Optional: drop rows with too many NAs
scorecard_state_year_clean <- scorecard_state_year_clean %>%
  filter(!is.na(grad_debt_mdn_w))

# Save for merge step
readr::write_csv(scorecard_state_year_clean, "scorecard_state_year_clean.csv")


YEAR,n
<int>,<int>
1996,59
1997,59
1998,59
1999,59
2001,59
2002,59
2003,59
2004,59
2005,59


YEAR,n,avail_debt,avail_price,avail_pell
<int>,<int>,<dbl>,<dbl>,<dbl>
1996,59,0.0000000,0.0000000,0
1997,59,0.9152542,0.0000000,0
1998,59,0.9152542,0.0000000,0
1999,59,0.9152542,0.0000000,0
2001,59,0.9152542,0.0000000,0
2002,59,0.9152542,0.0000000,0
2003,59,0.9152542,0.0000000,0
2004,59,0.9152542,0.0000000,0
2005,59,0.9152542,0.0000000,0


In [15]:
dim(scorecard_state_year_clean)

[1] 543  11

In [16]:
unique(scorecard_state_year_clean$YEAR)

[1] 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020

# 2. ACS cleanup
## ACS source file is downloaded from https://usa.ipums.org/usa/acs.shtml.
Steps:
1. register with gt email, and login.
2. On homepage, click "select data" on the top bar.
3. click "select samples", go to its page.
4. check the year box for ACS (2011 - 2020) and uncheck the content not desired. then click "submit sample selections" button at the bottom of the page.
5. select the variables, click the "plus"sign in front of the desired variable, then it will be added to the cart.
6. The quick way is using the "search" to find the desired variables and add to cart:
'YEAR''HHWT''STATEFIP''GQ''HHINCOME''PERNUM''PERWT''SEX''AGE''RACE''RACED''HISPAN''HISPAND''SCHOOL''EDUC''EDUCD''EMPSTAT''EMPSTATD''INCTOT''POVERTY'
7. After select the wanted variables, click "view cart" button.
8. Review the select vars, then click "create data extract"
9. IPUMS.org will review this extract request shortly.
10. Download the set once it's got approved.   

**See pdf version of the steps in  SourceFiles_download_instruction.pdf**


In [7]:
acs = read.csv("Raw_data/ACS_usa_00002.csv",header=TRUE) 

In [8]:
dim(acs)

[1] 44172608       20

In [9]:
colnames(acs)

[1] "YEAR"     "HHWT"     "STATEFIP" "GQ"       "HHINCOME" "PERNUM"  
 [7] "PERWT"    "SEX"      "AGE"      "RACE"     "RACED"    "HISPAN"  
[13] "HISPAND"  "SCHOOL"   "EDUC"     "EDUCD"    "EMPSTAT"  "EMPSTATD"
[19] "INCTOT"   "POVERTY"

In [10]:
head(acs,5)

,YEAR,HHWT,STATEFIP,GQ,HHINCOME,PERNUM,PERWT,SEX,AGE,RACE,RACED,HISPAN,HISPAND,SCHOOL,EDUC,EDUCD,EMPSTAT,EMPSTATD,INCTOT,POVERTY
,<int>,<dbl>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,2010,96,1,1,7500,1,96,2,75,1,100,0,0,1,6,63,3,30,7500,72
2,2010,97,1,1,30000,1,97,1,25,1,100,0,0,2,11,114,1,10,17000,172
3,2010,97,1,1,30000,2,128,2,26,1,100,0,0,2,7,71,1,10,13000,172
4,2010,97,1,1,30000,3,182,1,3,1,100,0,0,1,0,2,0,0,9999999,172
5,2010,90,1,1,29400,1,90,2,87,1,100,0,0,1,7,71,3,30,29400,283


In [15]:
# ==updated version=== add "asian","white","male" columns
# 1) Load  ACS microdata (2010–2023)
# acs <- read_csv("acs_df",
#              col_types = cols())
acs = read.csv("Raw_data/ACS_usa_00002.csv",header=TRUE)

# ---------- Helper functions ----------
# weighted mean (handles NAs)
wmean <- function(x, w) {
  ok <- is.finite(x) & is.finite(w) & w > 0
  if (!any(ok)) return(NA_real_)
  sum(x[ok] * w[ok]) / sum(w[ok])
}

# weighted share: mean of indicator with weights
wshare <- function(indicator, w) wmean(as.numeric(indicator), w)

# weighted median via cumulative weights
wmedian <- function(x, w) {
  ok <- is.finite(x) & is.finite(w) & w > 0
  if (!any(ok)) return(NA_real_)
  x <- x[ok]; w <- w[ok]
  ord <- order(x); x <- x[ord]; w <- w[ord]
  cw <- cumsum(w) / sum(w)
  x[ which(cw >= 0.5)[1] ]
}

# ---------- Basic coding ----------
# Exclude group quarters for household measures & person shares
acs <- acs %>% filter(GQ == 1)

# Mark HHINCOME sentinel as NA
acs <- acs %>%
  mutate(HHINCOME = if_else(HHINCOME >= 9999998, NA_integer_, HHINCOME))

# EDUCD thresholds (IPUMS detailed education):
# Bachelor's = 101; Master's = 114; Professional = 115; Doctorate = 116
is_ba_plus <- function(EDUCD) !is.na(EDUCD) & EDUCD >= 101

# Race/Ethnicity:
# RACE: 1 White, 2 Black, 4 Asian, ... ; Hispanic: HISPAN > 0
is_white   <- function(RACE)  !is.na(RACE)  & RACE == 1  # Added White
is_black   <- function(RACE)  !is.na(RACE)  & RACE == 2
is_asian   <- function(RACE)  !is.na(RACE)  & RACE == 4  # Added Asian
is_hisp    <- function(HISPAN) !is.na(HISPAN) & HISPAN > 0

# Unemployment proxy (from ACS EMPSTAT):
# EMPSTAT: 1 Employed, 2 Unemployed, others = NILF
is_emp    <- function(EMPSTAT) !is.na(EMPSTAT) & EMPSTAT == 1
is_unemp  <- function(EMPSTAT) !is.na(EMPSTAT) & EMPSTAT == 2

# In-school (SCHOOL codes: IPUMS: 1 no, 2 yes): treat 2 as enrolled
is_enrolled <- function(SCHOOL) !is.na(SCHOOL) & SCHOOL == 2

# Age bands
in_18_24 <- function(AGE) !is.na(AGE) & AGE >= 18 & AGE <= 24
in_25_34 <- function(AGE) !is.na(AGE) & AGE >= 25 & AGE <= 34

# Poverty thresholds (IPUMS POVERTY is % of poverty line, e.g., 172 = 172%)
below_100 <- function(POVERTY) is.finite(POVERTY) & POVERTY < 100
below_200 <- function(POVERTY) is.finite(POVERTY) & POVERTY < 200

# ---------- CPI adjustment hook ----------
# Provide a small CPI table with YEAR and CPI index (base = 2024 or 2025 = 100)
# Example placeholder (REPLACE with  CPI):
cpi <- tibble::tibble(
  YEAR = 2010:2023,
  CPI  = seq(80, 100, length.out = length(2010:2023))  # placeholder
)
# Merge CPI and deflate HH income to 2024 dollars
acs <- acs %>%
  left_join(cpi, by = c("YEAR" = "YEAR")) %>%
  mutate(HHINCOME_real = if_else(is.finite(HHINCOME) & is.finite(CPI),
                                 HHINCOME * (100 / CPI), NA_real_))

# ---------- Household-level median income (HHWT; one record per household) ----------
hh_income_state_year <- acs %>%
  filter(PERNUM == 1) %>%          # one person per household
  group_by(STATEFIP, YEAR) %>%
  summarise(
    median_hh_income_real = wmedian(HHINCOME_real, HHWT),
    .groups = "drop"
  )

# ---------- Person-level shares (PERWT) ----------
person_state_year <- acs %>%
  group_by(STATEFIP, YEAR) %>%
  summarise(
    # Education
    pct_ba_plus   = 100 * wshare(is_ba_plus(EDUCD), PERWT),

    # Demographics
    pct_male      = 100 * wshare(SEX == 1, PERWT), # ADDED: Male share
    pct_female    = 100 * wshare(SEX == 2, PERWT),
    pct_white     = 100 * wshare(is_white(RACE), PERWT),    # ADDED: White share
    pct_black     = 100 * wshare(is_black(RACE), PERWT),
    pct_asian     = 100 * wshare(is_asian(RACE), PERWT),    # ADDED: Asian share
    pct_hispanic  = 100 * wshare(is_hisp(HISPAN), PERWT),

    # Age composition
    pct_18_24     = 100 * wshare(in_18_24(AGE), PERWT),
    pct_25_34     = 100 * wshare(in_25_34(AGE), PERWT),

    # Poverty
    pct_poverty_lt100 = 100 * wshare(below_100(POVERTY), PERWT),
    pct_poverty_lt200 = 100 * wshare(below_200(POVERTY), PERWT),

    # Enrollment
    pct_enrolled_college = 100 * wshare(is_enrolled(SCHOOL), PERWT),

    # Unemployment proxy: unemployed / (employed + unemployed)
    unemp_rate_acs = {
      emp   <- wshare(is_emp(EMPSTAT), PERWT)
      unemp <- wshare(is_unemp(EMPSTAT), PERWT)
      if (is.na(emp) || is.na(unemp) || (emp + unemp) == 0) NA_real_
      else 100 * (unemp / (emp + unemp))
    },

    .groups = "drop"
  )

# ---------- Combine household & person aggregates ----------
acs_state_year <- hh_income_state_year %>%
  left_join(person_state_year, by = c("STATEFIP", "YEAR")) %>%
  arrange(STATEFIP, YEAR)

# (Optional) Map STATEFIP → USPS later when  merge to FSA/Scorecard.
# For now, keep keys as (STATEFIP, YEAR).
write_csv(acs_state_year, "acs_state_year_controls.csv")

# Quick peek
print(head(acs_state_year, 10), width = Inf)

# A tibble: 10 × 16
   STATEFIP  YEAR median_hh_income_real pct_ba_plus pct_male pct_female
      <int> <int>                 <dbl>       <dbl>    <dbl>      <dbl>
 1        1  2010                50000         15.5     48.3       51.7
 2        1  2011                49302.        15.8     47.9       52.1
 3        1  2012                49653.        16.6     48.0       52.0
 4        1  2013                50345.        16.9     48.2       51.8
 5        1  2014                49330.        16.5     48.2       51.8
 6        1  2015                50974.        17.3     48.1       51.9
 7        1  2016                51440.        17.8     48.2       51.8
 8        1  2017                52110.        18.4     48.2       51.8
 9        1  2018                53083.        18.4     48.1       51.9
10        1  2019                54344.        19.1     48.2       51.8
   pct_white pct_black pct_asian pct_hispanic pct_18_24 pct_25_34
       <dbl>     <dbl>     <dbl>        <dbl>     

#### This part includs the FSA files combine &cleanup , and 3 required  sources' csv files join.

# 3. FSA application volume datasets 

## download link:https://studentaid.gov/data-center/student/application-volume/fafsa-school-state
steps:
1. Go to page via above link.
2. On the page, find **"FAFSA by State"**
3. Under each year range, such as "2021-2022", click the blue link such as " 2021-2022 Q7", then xls will be downloaded automatically.
4. Click all the links of the desired years to download the data
5. Then use the code below to merge to 1 single csv file.

**See pdf version of the steps in  SourceFiles_download_instruction.pdf**


Pulls both Quarter Total and Cycle Year-to-Date Total blocks.

Normalizes state names → USPS codes (drops non-states like “Foreign Country”).

Derives calendar year for the quarter: YEAR_cal = FY - 1 for Q1, else FY.

Returns tidy rows: one per (state, cycle, fiscal-Q), with quarter_total, ytd_total and dependent/independent subtotals if  want them.

In [2]:
# ---- FAFSA Application Data by State (R tidy pipeline) -----------------------
library(readxl)
library(dplyr)
library(stringr)
library(tidyr)
library(purrr)
library(readr)

In [47]:
# --- Flexible discovery + runner (add below  helpers) --------------------

path <- "FSA"   # folder with the XLS/XLSX files

# extract cycle years from a filename like "...2018-2019..." (tolerates separators)
extract_cycle_yrs <- function(x) {
  m <- str_match(x, "(?i)(20\\d{2})\\D{0,2}(20\\d{2})")
  if (all(is.na(m))) return(c(NA_integer_, NA_integer_))
  c(as.integer(m[,2]), as.integer(m[,3]))
}

# extract a quarter token: q1..q7 or through-q# / thru-q#
extract_quarter <- function(x) {
  m <- str_match(x, "(?i)(?:^|[^a-z])(?:q(\\d)|through-q(\\d)|thru-q(\\d))")
  qnum <- suppressWarnings(as.integer(coalesce(m[,2], m[,3], m[,4])))
  ifelse(is.na(qnum), NA_character_, paste0("Q", qnum))
}

# find files for a given cycle (handles spaces/underscores/case, xls/xlsx, and 'through-q2')
find_cycle_files <- function(start, end) {
  files <- list.files(path, full.names = TRUE)
  base  <- basename(files)
  yrs   <- t(vapply(base, extract_cycle_yrs, integer(2)))
  qs    <- extract_quarter(base)

  tibble(file = files, base, start = yrs[,1], end = yrs[,2], quarter = qs) %>%
    filter(start == !!start, end == !!end, str_detect(base, "(?i)state")) %>%
    filter(!is.na(quarter)) %>%
    arrange(quarter)
}

#  full cycle list with max quarter per  table
cycles <- tibble::tribble(
  ~start, ~end, ~max_q,
  2011L, 2012L, 6L,
  2012L, 2013L, 6L,
  2013L, 2014L, 6L,
  2014L, 2015L, 5L,
  2015L, 2016L, 6L,
  2016L, 2017L, 6L,
  2017L, 2018L, 7L,
  2018L, 2019L, 7L,
  2019L, 2020L, 7L,
  2020L, 2021L, 7L,
  2021L, 2022L, 7L,
  2022L, 2023L, 7L,
  2023L, 2024L, 7L,
  2024L, 2025L, 7L,   # includes the special "through-q2" file; see filter below
  2025L, 2026L, 3L
)

# special-case filter for 2024-2025 to accept "...through-q2.xls" as Q2
normalize_special_q <- function(df){
  df %>%
    mutate(
      quarter = dplyr::case_when(
        str_detect(base, "(?i)through-q(\\d+)") ~ paste0("Q", str_match(base, "(?i)through-q(\\d+)")[,2]),
        TRUE ~ quarter
      )
    )
}

read_one_cycle_flex <- function(start, end, max_q) {
  cand <- find_cycle_files(start, end) %>% normalize_special_q()

  if (nrow(cand) == 0) {
    warning(sprintf("No files found for %d-%d.", start, end))
    return(tibble())
  }

  cand <- cand %>%
    mutate(qnum = as.integer(str_remove(quarter, "Q"))) %>%
    filter(qnum <= max_q) %>%
    arrange(qnum)

  message(sprintf("Reading %d files for %d-%d (up to Q%d):", nrow(cand), start, end, max_q))
  print(cand$base)

  out <- map_dfr(cand$file, ~ read_fafsa_state_file(.x, cycle_start_year = start))

  out_file <- sprintf("%s/fafsa_state_quarter_%d_%02d.csv", path, start, end %% 100)
  readr::write_csv(out, out_file)
  message(sprintf("  -> wrote %s (%d rows)", out_file, nrow(out)))
  out
}

# run all cycles & also write a combined file
fafsa_state_quarter_all <-
  pmap_dfr(cycles, ~ read_one_cycle_flex(..1, ..2, ..3))

if (nrow(fafsa_state_quarter_all) > 0) {
  readr::write_csv(fafsa_state_quarter_all, file.path(path, "fafsa_state_quarter_2011_12_to_2025_26.csv"))
  dplyr::count(fafsa_state_quarter_all, quarter) %>% print(n = 10)
}


Reading 6 files for 2011-2012 (up to Q6):



[1] "2011_2012_AppDatabyState_Q1.xls" "2011_2012_AppDatabyState_Q2.xls"
[3] "2011_2012_AppDatabyState_Q3.xls" "2011_2012_AppDatabyState_Q4.xls"
[5] "2011_2012_AppDatabyState_Q5.xls" "2011_2012_AppDatabyState_Q6.xls"


New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Stude

[1] "2012_2013_App_DatabyState_Q1.xls"   "2012_2013-App_Data_by_State_Q2.xls"
[3] "2012_2013-App_Data_by_State_Q3.xls" "2012_2013-App_Data_by_State_Q4.xls"
[5] "2012_2013-App_Data_by_State_Q5.xls" "2012_2013-App_Data_by_State_Q6.xls"


New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Stude

...

[1] "2024-2025-app-data-by-state-through-q2.xls"
[2] "2024-2025-app-data-by-state-q3.xls"        
[3] "2024-2025-app-data-by-state-q4.xls"        
[4] "2024-2025-app-data-by-state-q5.xls"        
[5] "2024-2025-app-data-by-state-q6.xls"        
[6] "2024-2025-app-data-by-state-q7.xls"        


New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Stude

[1] "2025-2026-app-data-by-state-q1.xls" "2025-2026-app-data-by-state-q2.xls"
[3] "2025-2026-app-data-by-state-q3.xls"


New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
New names:
• `Dependent Students` -> `Dependent Students...2`
• `Independent Students` -> `Independent Students...3`
• `Dependent Students` -> `Dependent Students...5`
• `Independent Students` -> `Independent Students...6`
  -> wrote FSA/fafsa_state_quarter_2025_26.csv (153 rows)



# A tibble: 8 × 2
  quarter     n
  <chr>   <int>
1 Q1        306
2 Q2        357
3 Q3        357
4 Q4        357
5 Q5        357
6 Q6        357
7 Q7        357
8 NA       2295


In [49]:
# --- Fix missing quarter from file names ---
library(dplyr)
library(stringr)
library(readr)

infile  <- "fafsa_state_quarter_2011_12_to_2025_26.csv"
outfile <- "fafsa_state_quarter_2011_12_to_2025_26_FIXED.csv"

# helper: pull Q1..Q7 from any filename style (e.g., ...Q1.xls or ...-q1.xls)
q_from_file <- function(x) {
  m <- str_match(x, "(?i)q\\s*([1-7])")[, 2]      # capture the digit 1–7
  out <- ifelse(is.na(m), NA_character_, paste0("Q", m))
  toupper(out)
}

df <- read_csv(infile, show_col_types = FALSE) %>%
  mutate(
    quarter = if_else(is.na(quarter) | quarter == "" | quarter == "NA",
                      q_from_file(file),
                      quarter)
  )

# quick sanity: how many quarters still missing?
sum(is.na(df$quarter))

write_csv(df, outfile)
cat("Wrote:", outfile, "\n")


[1] 0

Wrote: fafsa_state_quarter_2011_12_to_2025_26_FIXED.csv 


In [51]:
df


state,cycle_start,quarter,FY,YEAR_cal,quarter_total,ytd_total,file
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
AK,2011,Q1,2011,2011,16199,16199,2011_2012_AppDatabyState_Q1.xls
AL,2011,Q1,2011,2011,99958,99958,2011_2012_AppDatabyState_Q1.xls
AR,2011,Q1,2011,2011,60437,60437,2011_2012_AppDatabyState_Q1.xls
AZ,2011,Q1,2011,2011,142551,142551,2011_2012_AppDatabyState_Q1.xls
CA,2011,Q1,2011,2011,1357976,1357976,2011_2012_AppDatabyState_Q1.xls
CO,2011,Q1,2011,2011,132820,132820,2011_2012_AppDatabyState_Q1.xls
CT,2011,Q1,2011,2011,110995,110995,2011_2012_AppDatabyState_Q1.xls
DC,2011,Q1,2011,2011,15985,15985,2011_2012_AppDatabyState_Q1.xls
DE,2011,Q1,2011,2011,20395,20395,2011_2012_AppDatabyState_Q1.xls


In [52]:
dim(df)

[1] 4743    8

#### annualize the FAFSA series so it lines up with the ACS and Scorecard state–year tables.

converts quarter to a number,

picks the final YTD observation for each state × cycle,

maps the cycle to its expected last quarter (so we can flag incomplete cycles like 2025–26),

outputs a tidy state–year table keyed by state and YEAR (the cycle’s ending year).

In [167]:
library(readr)
library(dplyr)
library(stringr)

# 1) read the cleaned quarterly file
df <- read_csv("FSA_cleaned/fafsa_state_quarter_2011_12_to_2025_26_FIXED.csv",
               show_col_types = FALSE)

# 2) helper: Q1..Q7 -> numeric
q_num <- function(q) as.integer(str_remove(q, "^Q"))

df2 <- df %>%
  mutate(qtr_num = q_num(quarter))

# 3) expected final quarter by cycle (from  list)
exp_last_q <- tibble::tibble(
  cycle_start = c(2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025),
  exp_q_last  = c(6,6,6,5,6,6,7,7,7,7,7,7,7,7,3)
)

# 4) choose the final YTD per state×cycle (largest quarter reported),
#    attach the expected Q so we can flag completeness
fsa_state_year <- df2 %>%
  group_by(state, cycle_start) %>%
  arrange(qtr_num, .by_group = TRUE) %>%
  slice_tail(n = 1) %>%                               # keep last-reported quarter
  ungroup() %>%
  left_join(exp_last_q, by = "cycle_start") %>%
  mutate(
    year = cycle_start,                                   # annual key = cycle ending year
    is_final_cycle = qtr_num >= exp_q_last             # TRUE if we reached the expected last quarter
  ) %>%
  transmute(
    state,
    year,                                              # join key to ACS/Scorecard
    fafsa_apps_total = ytd_total,                      # annualized FAFSA apps (YTD at last quarter)
    last_reported_quarter = quarter,                   # e.g., "Q7"
    expected_last_quarter = paste0("Q", exp_q_last),   # reference
    is_final_cycle,
    source_file = file                                  # lineage
  ) %>%
  arrange(state, year)

# 5) quick sanity checks (optional)
# dplyr::count(fsa_state_year, YEAR, is_final_cycle)
# summary(fsa_state_year$fafsa_apps_total)

# 6) save
write_csv(fsa_state_year, "fsa_state_year_annualized_2.csv")


In [169]:
# Gets a single-column data frame of unique years
distinct_years <- fsa_state_year %>%
  distinct(year)

# To view the result:
print(distinct_years)

# A tibble: 15 × 1
    year
   <dbl>
 1  2011
 2  2012
 3  2013
 4  2014
 5  2015
 6  2016
 7  2017
 8  2018
 9  2019
10  2020
11  2021
12  2022
13  2023
14  2024
15  2025


In [168]:
dim(fsa_state_year)

[1] 765   7

In [56]:
fsa_state_year

state,YEAR,fafsa_apps_total,last_reported_quarter,expected_last_quarter,is_final_cycle,source_file
<chr>,<dbl>,<dbl>,<chr>,<chr>,<lgl>,<chr>
AK,2011,41438,Q6,Q6,TRUE,2011_2012_AppDatabyState_Q6.xls
AK,2012,41694,Q6,Q6,TRUE,2012_2013-App_Data_by_State_Q6.xls
AK,2013,41205,Q6,Q6,TRUE,2013_2014-App_Data_by_State_Q6.xls
AK,2014,38836,Q5,Q5,TRUE,2014_2015-App_Data_by_State_Q5.xls
AK,2015,38537,Q6,Q6,TRUE,2015_2016-App_Data_By_State_Q6.xls
AK,2016,36545,Q6,Q6,TRUE,2016_2017-App_Data_by_State_Q6.xls
AK,2017,37290,Q7,Q7,TRUE,2017_2018-App_Data_by_State_Q7.xls
AK,2019,36050,Q7,Q7,TRUE,2018-2019-app-data-by-state-q7.xls
AK,2020,33893,Q7,Q7,TRUE,2019-2020-app-data-by-state-q7.xls


In [57]:
fsa_state_year_false <- fsa_state_year %>%
  filter(is_final_cycle == FALSE)
fsa_state_year_false

state,YEAR,fafsa_apps_total,last_reported_quarter,expected_last_quarter,is_final_cycle,source_file
<chr>,<dbl>,<dbl>,<chr>,<chr>,<lgl>,<chr>


In [59]:
yr_cnt <- fsa_state_year%>% 
            group_by(state)%>%
            summarise(
                Count_of_Records =n()
                )

yr_cnt

state,Count_of_Records
<chr>,<int>
AK,15
AL,15
AR,15
AZ,15
CA,15
CO,15
CT,15
DC,15
DE,15


## 4. Combine all 3 sets, 2011-2020 
#### For project analysis

In [81]:
# path = "data_cleaned"

fsa <- read_csv("data_cleaned/fsa_state_year_annualized.csv",show_col_types = FALSE)
acs <- read_csv("data_cleaned/acs_state_year_controls.csv",show_col_types = FALSE)
sc <- read_csv("data_cleaned/scorecard_state_year.csv",show_col_types = FALSE)


In [106]:
list.files("data_cleaned")

[1] "acs_state_year_controls.csv"   "fsa_state_year_annualized.csv"
[3] "scorecard_state_year.csv"

In [ ]:
# ===============================
# Combine 3 source csv files.
# ===============================
# ---- Libraries ----
library(vroom)
library(dplyr)
library(stringr)
options(dplyr.summarise.inform = FALSE)

# ---- Helpers ----
to_num <- function(x) suppressWarnings(as.numeric(gsub("[^0-9.-]", "", x)))

# FIPS -> USPS (50 states + DC)
fips_usps <- tibble::tibble(
  statefip = c(1,2,4,5,6,8,9,10,11,12,13,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,44,45,46,47,48,49,50,51,53,54,55,56),
  state    = c("AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY")
)
# =====================================================================
# ---- Paths (relative) udpate the path accordingly to real situation----
sc_path  <- "data_cleaned/scorecard_state_year.csv"
acs_path <- "data_cleaned/acs_state_year_controls.csv"
fsa_path <- "data_cleaned/fsa_state_year_annualized.csv"
# =====================================================================

# ---- SCORECARD (already at state–year grain) ----
scorecard <- vroom::vroom(sc_path, show_col_types = FALSE) %>%
  { setNames(., tolower(names(.))) } %>%
  select(
    state, year,
    log_grad_debt_state, grad_debt_mdn_w, state_net_price, state_pell_share,
    sector_public_share, sector_private_np_share, sector_forprofit_share,
    state_stem_share, ugds_total
  ) %>%
  mutate(state = toupper(state), year = as.integer(year)) %>%
  filter(year >= 2011, year <= 2020) %>%
  distinct(state, year, .keep_all = TRUE)

# ---- ACS (robust to state/statefip presence) ----
acs_raw <- vroom::vroom(acs_path, show_col_types = FALSE)
names(acs_raw) <- tolower(names(acs_raw))

if (!("state" %in% names(acs_raw)) && !("statefip" %in% names(acs_raw))) {
  stop("ACS file lacks both `state` and `statefip`. Please include one to join by state.")
}

acs <- acs_raw %>%
  # If only statefip exists, map to USPS
  {
    if ("state" %in% names(.)) {
      mutate(., state = toupper(state))
    } else {
      mutate(., statefip = as.integer(statefip)) %>%
        inner_join(fips_usps, by = "statefip") %>%
        select(-statefip)
    }
  } %>%
  select(
    state, year,
    median_hh_income_real, pct_ba_plus, pct_male,pct_female, pct_black, pct_hispanic,pct_white,pct_asian,
    pct_18_24, pct_25_34, pct_poverty_lt100, pct_poverty_lt200,
    pct_enrolled_college, unemp_rate_acs
  ) %>%
  mutate(year = as.integer(year)) %>%
  filter(year >= 2011, year <= 2020) %>%
  distinct(state, year, .keep_all = TRUE)

# ---- FAFSA (annualized state–year apps) ----
fsa <- vroom::vroom(fsa_path, show_col_types = FALSE) %>%
  { setNames(., tolower(names(.))) } %>%
  select(state, year, fafsa_apps_total,
         last_reported_quarter, expected_last_quarter,
         is_final_cycle, source_file) %>%
  mutate(
    state = toupper(state),
    year  = as.integer(year),
    fafsa_apps_total = to_num(fafsa_apps_total)
  ) %>%
  filter(year >= 2011, year <= 2020) %>%
  group_by(state, year) %>%
  slice_max(order_by = last_reported_quarter, with_ties = FALSE) %>%
  ungroup() %>%
  distinct(state, year, .keep_all = TRUE)

# ---- 3-way INNER JOIN (state, year) ----
joined <- scorecard %>%
  inner_join(acs, by = c("state","year")) %>%
  inner_join(fsa, by = c("state","year")) %>%
  arrange(state, year)

# ---- Save (Ensure standard CSV format) ----
out_path <- "data_cleaned/state_year_joined_2011_2020.csv"
vroom::vroom_write(joined, out_path,delim = ",") # Explicitly set delimiter)



message("Done. Rows: ", nrow(joined), " | Saved to: ", out_path)


In [173]:
library(dplyr)
library(naniar)

# df <- readr::read_csv("data_cleaned/state_year_joined_2011_2020.csv", show_col_types = FALSE)
count(joined, year)
count(joined, state)
miss_var_summary(joined) %>% print(n=20)


year,n
<int>,<int>
2011,51
2012,51
2013,51
2014,51
2015,51
2016,51
2017,51
2018,51
2019,51


state,n
<chr>,<int>
AK,10
AL,10
AR,10
AZ,10
CA,10
CO,10
CT,10
DC,10
DE,10


# A tibble: 27 × 3
   variable                n_miss pct_miss
   <chr>                    <int>    <num>
 1 state                        0        0
 2 year                         0        0
 3 log_grad_debt_state          0        0
 4 grad_debt_mdn_w              0        0
 5 state_net_price              0        0
 6 state_pell_share             0        0
 7 sector_public_share          0        0
 8 sector_private_np_share      0        0
 9 sector_forprofit_share       0        0
10 state_stem_share             0        0
11 ugds_total                   0        0
12 median_hh_income_real        0        0
13 pct_ba_plus                  0        0
14 pct_female                   0        0
15 pct_black                    0        0
16 pct_hispanic                 0        0
17 pct_18_24                    0        0
18 pct_25_34                    0        0
19 pct_poverty_lt100            0        0
20 pct_poverty_lt200            0        0
# ℹ 7 more rows
